# KAUST Project 9 — ZeroWaste-f *external-domain crop evaluation*
**No model training. No Google Drive access. Requires your own licensed, labeled ZeroWaste-f crop sample**.

The uploaded ZIP must have image crops arranged in one folder per original annotated material:
```
zerowaste_labeled_crops.zip
  cardboard/frame001_obj1.jpg
  soft_plastic/frame002_obj3.jpg
  rigid_plastic/frame003_obj2.jpg
  metal/frame004_obj7.jpg
```
Labels may be spaced or underscored (`soft plastic`, `rigid_plastic`). Crops must be made from **verified ground-truth annotations**, not guessed from raw frames. A separate `Create_ZeroWaste_Crops_From_COCO.py` helper is included for COCO annotations.

**Important:** This is a cross-domain *classification* test, not Phase 2 detection or segmentation training. We merge the two ZeroWaste plastic subclasses into the TrashNet `plastic` class for comparison; we report full six-way classifier misses separately from the restricted three-class diagnostic. Without valid labeled crops, no ZeroWaste score can be claimed.

In [ ]:
import torch, torchvision
from torchvision import models,transforms
from PIL import Image
import json,zipfile,os,io,collections,numpy as np,pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from google.colab import files
print('Torch',torch.__version__,'Torchvision',torchvision.__version__)

In [ ]:
print('1) Upload saved best_model.pt and 2) annotated crop archive ZIP')
uploaded=files.upload()
pts=[Path(f) for f in uploaded if f.lower().endswith('.pt')]
zz=[Path(f) for f in uploaded if f.lower().endswith('.zip')]
if len(pts)!=1 or len(zz)!=1: raise ValueError('Upload one .pt checkpoint and one labeled-crops .zip')
ck=torch.load(pts[0],map_location='cpu',weights_only=False)
if ck.get('kind')!='finetune': raise ValueError('Expected best fine-tuned ResNet18 model')
classes=ck['classes'];n=len(classes)
model=models.resnet18(weights=None);model.fc=torch.nn.Linear(model.fc.in_features,n)
model.load_state_dict(ck['state_dict'],strict=True);model.eval()
transform=transforms.Compose([transforms.Resize((ck['img_size'],ck['img_size'])),transforms.ToTensor(),
 transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
print('Checkpoint loaded; classes:',classes)

In [ ]:
accepted={'cardboard':'cardboard','metal':'metal',
 'soft_plastic':'plastic','soft plastic':'plastic','rigid_plastic':'plastic','rigid plastic':'plastic'}
rows=[]
with zipfile.ZipFile(zz[0]) as archive:
 for entry in archive.infolist():
    if entry.is_dir() or not entry.filename.lower().endswith(('.jpg','.jpeg','.png')): continue
    parts=Path(entry.filename).parts
    label_from_folder=next((s.lower() for s in parts[:-1] if s.lower() in accepted),None)
    if label_from_folder is None: continue
    try:
        img=Image.open(io.BytesIO(archive.read(entry))).convert('RGB')
    except Exception as e:
        print('Skipping damaged crop',entry.filename,e);continue
    if min(img.size)<16: continue
    with torch.no_grad():
        prob=torch.softmax(model(transform(img).unsqueeze(0)),dim=1).squeeze(0).numpy()
    raw_pred=classes[int(prob.argmax())]
    three_classes=['cardboard','metal','plastic']
    restricted_pred=three_classes[int(np.argmax([prob[classes.index(k)] for k in three_classes]))]
    rows.append({'file':entry.filename,'source_class':label_from_folder,
                 'true_merged':accepted[label_from_folder],
                 'predicted_six_way':raw_pred,'predicted_restricted_three_way':restricted_pred,
                 'confidence_six_way':round(float(prob.max()),4)})
df=pd.DataFrame(rows)
if df.empty:raise ValueError('No valid labelled crops found; check folder names.')
df.to_csv('zerowaste_crop_predictions.csv',index=False)
true=df['true_merged'].tolist();pred=df['predicted_six_way'].tolist();pred3=df['predicted_restricted_three_way'].tolist()
labels=['cardboard','metal','plastic']
summary={'n_valid_labeled_crops':len(df),
 'source_class_distribution':df.source_class.value_counts().to_dict(),
 'full_six_class_predictions_accuracy_on_three_mapped_true_categories':float(accuracy_score(true,pred)),
 'full_six_class_predictions_macro_f1_target_3':float(f1_score(true,pred,labels=labels,average='macro',zero_division=0)),
 'restricted_three_class_accuracy_diagnostic_only':float(accuracy_score(true,pred3)),
 'restricted_three_class_macro_f1_diagnostic_only':float(f1_score(true,pred3,labels=labels,average='macro',zero_division=0)),
 'class_mapping':{'cardboard':'cardboard','metal':'metal','soft_plastic':'plastic','rigid_plastic':'plastic'},
 'status':'Real cross-domain external test only if crop labels derive from actual ZeroWaste annotations; not a detector evaluation.',
 'limitations':['Class names and taxonomies differ','Small crops may be correlated by conveyor frame','Dataset license must be respected','No expected performance thresholds supplied']}
Path('zerowaste_crop_metrics.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Full-six-way output report:')
print(classification_report(true,pred,labels=labels,zero_division=0))
files.download('zerowaste_crop_predictions.csv');files.download('zerowaste_crop_metrics.json')